1. Load Models and Preprocessing

In [ ]:
import pickle
import torch
from torchvision import models, transforms
from PIL import Image
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
from collections import Counter

# Load SVM model and vectorizer
with open('trained_classical_model.pkl', 'rb') as file:
    svm_model = pickle.load(file)

with open('vectorizer.pkl', 'rb') as file:
    vectorizer = pickle.load(file)

# Load VGG model
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg_model = models.vgg16(weights=None)
num_ftrs = vgg_model.classifier[6].in_features
vgg_model.classifier[6] = torch.nn.Linear(num_ftrs, 27)
vgg_model.load_state_dict(torch.load('vgg16_transfer_model.pth'))
vgg_model = vgg_model.to(device)
vgg_model.eval()

# Placeholder for BERT model
bert_model = None  # Replace with your trained BERT model

In [ ]:
# VGG prediction functions
vgg_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def vgg_predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = vgg_preprocess(image).unsqueeze(0).to(device)
    outputs = vgg_model(image_tensor)
    _, preds = torch.max(outputs, 1)
    return preds.item()

def vgg_predict_proba(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = vgg_preprocess(image).unsqueeze(0).to(device)
    outputs = vgg_model(image_tensor)
    probabilities = torch.nn.functional.softmax(outputs, dim=1)
    return probabilities.cpu().detach().numpy()[0]

# SVM prediction functions
def svm_predict(text):
    vectorized_text = vectorizer.transform([text])
    return svm_model.predict(vectorized_text)[0]

def svm_predict_proba(text):
    vectorized_text = vectorizer.transform([text])
    return svm_model.predict_proba(vectorized_text)[0]

2. Define the Ensemble Classifier

In [ ]:
class VotingEnsemble:
    def __init__(self, classifiers, voting='hard'):
        self.classifiers = classifiers
        self.voting = voting

    def predict(self, X_text, X_image):
        if self.voting == 'hard':
            predictions = []
            for name, clf in self.classifiers:
                if name == 'svm':
                    predictions.append(svm_predict(X_text))
                elif name == 'bert':
                    predictions.append(clf.predict(X_text))  # Replace with BERT prediction logic
                elif name == 'vgg':
                    predictions.append(vgg_predict(X_image))
            vote_counts = Counter(predictions)
            return vote_counts.most_common(1)[0][0]

        elif self.voting == 'soft':
            probabilities = []
            for name, clf in self.classifiers:
                if name == 'svm':
                    probabilities.append(svm_predict_proba(X_text))
                elif name == 'bert':
                    probabilities.append(clf.predict_proba(X_text))  # Replace with BERT probability logic
                elif name == 'vgg':
                    probabilities.append(vgg_predict_proba(X_image))
            avg_proba = np.mean(probabilities, axis=0)
            return np.argmax(avg_proba)

3. Initialize the Ensemble

In [ ]:
# Initialize the ensemble classifiers
hard_voting_ensemble = VotingEnsemble(
    classifiers=[('svm', svm_model), ('bert', bert_model), ('vgg', vgg_model)],
    voting='hard'
)

soft_voting_ensemble = VotingEnsemble(
    classifiers=[('svm', svm_model), ('bert', bert_model), ('vgg', vgg_model)],
    voting='soft'
)

Evaluate Hard Voting

In [ ]:
# Predict using hard voting
hard_predictions = [hard_voting_ensemble.predict(text, image) for text, image in zip(X_text_test, X_image_test)]

# Evaluate hard voting
print("Hard Voting Performance:")
print(classification_report(y_test, hard_predictions))
print("Accuracy:", accuracy_score(y_test, hard_predictions))

Evaluate Soft Voting


In [ ]:
# Predict using soft voting
soft_predictions = [soft_voting_ensemble.predict(text, image) for text, image in zip(X_text_test, X_image_test)]

# Evaluate soft voting
print("Soft Voting Performance:")
print(classification_report(y_test, soft_predictions))
print("Accuracy:", accuracy_score(y_test, soft_predictions))

## MODEL WITHOUT BERT

In [5]:
import pickle
import torch
from torchvision import models, transforms
from PIL import Image
import numpy as np
import sklearn
from sklearn.metrics import classification_report, accuracy_score
from collections import Counter

# Load SVM model and vectorizer
with open('trained_classical_model.pkl', 'rb') as file:
    svm_model = pickle.load(file)

with open('vectorizer.pkl', 'rb') as file:
    vectorizer = pickle.load(file)

# Load VGG model
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg_model = models.vgg16(weights=None)
num_ftrs = vgg_model.classifier[6].in_features
vgg_model.classifier[6] = torch.nn.Linear(num_ftrs, 27)
vgg_model.load_state_dict(torch.load('vgg16_transfer_model.pth'))
vgg_model = vgg_model.to(device)
vgg_model.eval()

FileNotFoundError: [Errno 2] No such file or directory: 'vgg16_transfer_model.pth'

In [ ]:
# VGG prediction functions
vgg_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def vgg_predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = vgg_preprocess(image).unsqueeze(0).to(device)
    outputs = vgg_model(image_tensor)
    _, preds = torch.max(outputs, 1)
    return preds.item()

def vgg_predict_proba(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = vgg_preprocess(image).unsqueeze(0).to(device)
    outputs = vgg_model(image_tensor)
    probabilities = torch.nn.functional.softmax(outputs, dim=1)
    return probabilities.cpu().detach().numpy()[0]

# SVM prediction functions
def svm_predict(text):
    vectorized_text = vectorizer.transform([text])
    return svm_model.predict(vectorized_text)[0]

def svm_predict_proba(text):
    vectorized_text = vectorizer.transform([text])
    return svm_model.predict_proba(vectorized_text)[0]

In [ ]:
class VotingEnsemble:
    def __init__(self, classifiers, voting='hard'):
        self.classifiers = classifiers
        self.voting = voting

    def predict(self, X_text, X_image):
        if self.voting == 'hard':
            predictions = []
            for name, clf in self.classifiers:
                if name == 'svm':
                    predictions.append(svm_predict(X_text))
                elif name == 'vgg':
                    predictions.append(vgg_predict(X_image))  # Removed BERT logic
            vote_counts = Counter(predictions)
            return vote_counts.most_common(1)[0][0]

        elif self.voting == 'soft':
            probabilities = []
            for name, clf in self.classifiers:
                if name == 'svm':
                    probabilities.append(svm_predict_proba(X_text))
                elif name == 'vgg':
                    probabilities.append(vgg_predict_proba(X_image))  # Removed BERT logic
            avg_proba = np.mean(probabilities, axis=0)
            return np.argmax(avg_proba)

In [ ]:
# Initialize the ensemble classifiers without BERT
hard_voting_ensemble = VotingEnsemble(
    classifiers=[('svm', svm_model), ('vgg', vgg_model)],  # Removed BERT
    voting='hard'
)

soft_voting_ensemble = VotingEnsemble(
    classifiers=[('svm', svm_model), ('vgg', vgg_model)],  # Removed BERT
    voting='soft'
)

In [ ]:
# Predict using hard voting
hard_predictions = [hard_voting_ensemble.predict(text, image) for text, image in zip(X_text_test, X_image_test)]

# Evaluate hard voting
print("Hard Voting Performance:")
print(classification_report(y_test, hard_predictions))
print("Accuracy:", accuracy_score(y_test, hard_predictions))

In [ ]:
# Predict using soft voting
soft_predictions = [soft_voting_ensemble.predict(text, image) for text, image in zip(X_text_test, X_image_test)]

# Evaluate soft voting
print("Soft Voting Performance:")
print(classification_report(y_test, soft_predictions))
print("Accuracy:", accuracy_score(y_test, soft_predictions))